<table width="100%" bgcolor="#f4f5f6" style="width: 100%; background-color: #f4f5f6; border-left: 6px solid #ff9900; border-radius: 8px; border-collapse: collapse;">
  <tr>
    <td width="110" align="center" valign="middle" style="padding: 15px;">
      <img src="https://github.com/sansalamanca/shining-path_DP700/raw/main/data/logo%20libro.jpg" width="85" style="border-radius: 8px; display: block;">
    </td>
    <td valign="middle" style="padding: 15px 20px 15px 5px; font-family: Arial, sans-serif;">
      <p style="margin: 0 0 4px 0; font-size: 13px; color: #555555;">Excerpt from the Amazon book</p>
      <h2 style="margin: 4px 0 10px 0; font-size: 20px; font-weight: 900; color: #111111; border: none; padding: 0;">THE SHINING PATH TO DATA ENGINEER CERTIFICATE</h2>
      <p style="margin: 0; font-size: 14px; font-weight: bold;">
        <a href="https://www.amazon.com" target="_blank" style="color: #b12704; text-decoration: underline;">🛒 Buy on Amazon &rarr;</a>
      </p>
    </td>
  </tr>
</table>


In [7]:
from pyspark.sql.functions import col, dense_rank
from pyspark.sql.window import Window
from delta.tables import DeltaTable

silver_table_path = "shining-path_DP700/Lakehouse/Tables/fact_sales_silver"
dim_customer_path = "shining-path_DP700/Lakehouse/Tables/dim_customer_gold"
fact_sales_gold_path = "shining-path_DP700/Lakehouse/Tables/fact_sales_gold"

# 0. Asegurar que la tabla Silver exista (creándola al vuelo desde el CSV si no está presente)
if not spark.catalog.tableExists("temp_silver_check"):
    df_seed = spark.read.format("csv") \
        .option("header", "true") \
        .option("escape", '"') \
        .load("shining-path_DP700/Lakehouse/Files/sales.csv")
    df_seed.write.format("delta").mode("overwrite").save(silver_table_path)

# 1. Read data from the Silver layer
df_silver_sales = spark.read.format("delta").load(silver_table_path)

# 2. Create Customer Dimension (Type 1 SCD emulation) with a Surrogate Key
df_dim_customer = df_silver_sales.select("CustomerName", "EmailAddress") \
    .dropDuplicates(["CustomerName"]) \
    .withColumn("dim_customer_sk", dense_rank().over(Window.orderBy("CustomerName")))

df_dim_customer.write.format("delta").mode("overwrite").save(dim_customer_path)
print("Customer Dimension (Gold) created successfully.")

# 3. Create Fact Table by joining Silver data with the Customer Dimension
df_fact_sales = df_silver_sales.alias("s") \
    .join(df_dim_customer.alias("c"), col("s.CustomerName") == col("c.CustomerName"), "inner") \
    .select(
        col("s.SalesOrderNumber"),
        col("s.SalesOrderLineNumber"),
        col("c.dim_customer_sk"),
        col("s.OrderDate"),
        col("s.Item"),
        col("s.Quantity"),
        col("s.UnitPrice"),
        col("s.TaxAmount")
    )

df_fact_sales.write.format("delta").mode("overwrite").save(fact_sales_gold_path)
print("Fact Sales Table (Gold) created successfully with Surrogate Keys!")

Customer Dimension (Gold) created successfully.
Fact Sales Table (Gold) created successfully with Surrogate Keys!


In [8]:
# Read back and inspect both Gold tables
print("--- DIMENSION: dim_customer_gold ---")
df_dim_result = spark.read.format("delta").load(dim_customer_path)
df_dim_result.show(10, truncate=False)

print("--- FACT TABLE: fact_sales_gold ---")
df_fact_result = spark.read.format("delta").load(fact_sales_gold_path)
df_fact_result.show(10, truncate=False)
df_fact_result.printSchema()

--- DIMENSION: dim_customer_gold ---
+-------------+---------------------------+---------------+
|CustomerName |EmailAddress               |dim_customer_sk|
+-------------+---------------------------+---------------+
|Aaron Adams  |aaron48@adventure-works.com|1              |
|Aaron Allen  |aaron55@adventure-works.com|2              |
|Aaron Bryant |aaron17@adventure-works.com|3              |
|Aaron Chen   |aaron25@adventure-works.com|4              |
|Aaron Collins|aaron35@adventure-works.com|5              |
|Aaron Diaz   |aaron21@adventure-works.com|6              |
|Aaron Edwards|aaron34@adventure-works.com|7              |
|Aaron Evans  |aaron33@adventure-works.com|8              |
|Aaron Flores |aaron11@adventure-works.com|9              |
|Aaron Foster |aaron15@adventure-works.com|10             |
+-------------+---------------------------+---------------+
only showing top 10 rows

--- FACT TABLE: fact_sales_gold ---
+----------------+--------------------+---------------+------